# Practice Assessment 9: The Franchise Paradox

**Topic**: Hierarchical Models (Partial Pooling)
**Chapter**: 09
**Goal**: Understand how to use hierarchical modeling to make better estimates for groups with small sample sizes.

---

## 1. Scenario: The New Franchise Locations

You are the lead data scientist for a rapidly growing coffee shop franchise, *BayesBrewe*. The company has recently opened 10 new locations scattered across the country. 

The operational team needs to decide which shops are underperforming and require intervention (or closure), and which are promising and deserve more marketing budget. However, there is a problem:
- These shops opened at different times during the last month.
- Some shops have been open for nearly 30 days (plenty of data).
- Others opened just a few days ago (very little data).

You have daily revenue data for each shop. The management has a simple rule: **Any shop with a true long-term average daily revenue below $1200 is considered "Critical".**

Your job is to estimate the `true_average_revenue` for each of the 10 shops and compute the probability that they are in the "Critical" zone.

## 2. The Data

Run the cell below to generate the synthetic data. 

**Note**: 
- `shop_id`: The ID of the shop (0 to 9).
- `revenue`: The daily revenue in USD.

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

np.random.seed(84) # Fixed seed for reproducibility

# --- Generative Process (Hidden from the analyst in real life) ---
MU_GLOBAL_TRUE = 1500  # The average franchise makes $1500/day
TAU_TRUE = 300         # Standard deviation between shops (some are great, some bad)
SIGMA_TRUE = 200       # Daily variation within a shop
N_SHOPS = 10

# Randomly decide how many days each shop has been open (between 3 and 30 days)
sample_sizes = np.random.randint(3, 30, size=N_SHOPS)
# Let's force Shop 0 to have very few days to make the point clear
sample_sizes[0] = 3

# Generate the "True" average revenue for each shop
true_shop_means = np.random.normal(MU_GLOBAL_TRUE, TAU_TRUE, size=N_SHOPS)

# Generate daily observations
data_rows = []
for s_id in range(N_SHOPS):
    # Generate 'n' days of revenue for this shop
    revs = np.random.normal(true_shop_means[s_id], SIGMA_TRUE, size=sample_sizes[s_id])
    for r in revs:
        data_rows.append({'shop_id': s_id, 'revenue': r})

df = pd.DataFrame(data_rows)

print(f"Total observations: {len(df)}")
print("Sample of data:")
print(df.sample(5).sort_values('shop_id'))

# Summary of sample sizes per shop
print("\nDays open per shop:")
print(df.groupby('shop_id').size())

---

## Task 1: The Naive Approaches (No Pooling & Complete Pooling)

Before building the Bayesian model, let's look at the raw data.

1.  **No Pooling**: Calculate the sample mean revenue for each shop independently.
2.  **Complete Pooling**: Calculate the global mean revenue of all data combined.

**Objective**: Create a DataFrame `summary_df` that contains `n_days`, `raw_mean` for each shop.

In [ ]:
# Your code here
# summary_df = ...


### Question 1
Look at the `raw_mean` for **Shop 0** (which has only 3 days of data). 
- If this mean is very high (e.g., $2000), do you trust it? 
- If it is very low (e.g., $900), should we immediately close the shop?
- Why is the sample mean unreliable here?

*(Write your answer here)*

---

## Task 2: The Hierarchical (Partial Pooling) Model

Now, construct a Hierarchical Bayesian Model to estimate the average revenue for each shop.

**Model Structure**:
- **Level 1 (Observation)**: The revenue $y$ on a given day for shop $j$ comes from a Normal distribution centered at that shop's true mean $\theta_j$.
  $$ y_{ij} \sim \text{Normal}(\theta_{j}, \sigma_{within}) $$
- **Level 2 (Population)**: The true means of the shops $\theta_j$ are not unrelated; they come from a population of shops.
  $$ \theta_j \sim \text{Normal}(\mu_{global}, \tau_{between}) $$

**Priors**:
- Choose reasonable weakly informative priors for $\mu_{global}$, $\tau_{between}$, and $\sigma_{within}$.
- *Hint*: Revenue is positive and likely in the thousands. $\sigma$ must be positive.

**Implementation**:
- Use `pymc`. 
- Use coordinates/dimensions for `shop_id` to make the code clean.
- Sample from the posterior.

In [ ]:
shop_idx = df['shop_id'].values
n_shops = len(df['shop_id'].unique())

with pm.Model() as hierarchical_model:
    # --- Your Code Here ---
    # 1. Priors for population parameters (mu_global, tau_between)
    
    # 2. Prior for within-shop variation (sigma_within)

    # 3. Shop-specific estimates (theta)
    # Hint: theta = pm.Normal(..., shape=n_shops)

    # 4. Likelihood
    
    # 5. Inference
    # trace = pm.sample(...)
    pass

## Task 3: Analyzing Shrinkage

Extract the posterior mean for each shop's $\theta_j$ from your trace. Add these as a new column `bayes_mean` to your `summary_df`.

Create a plot that compares the `raw_mean` vs `bayes_mean`.
- Plot the `raw_mean` on the x-axis.
- Plot the `bayes_mean` on the y-axis.
- Add a diagonal line ($y=x$).
- *Bonus*: Draw arrows showing the movement from raw to Bayes estimates.

In [ ]:
# Your code here


### Question 2
Which shops experienced the largest "shrinkage" (difference between raw and Bayes means)? Why did the model move their estimates so much?

*(Write your answer here)*

---

## Task 4: Decision Making under Uncertainty

Recalling the management rule: **"Critical" if True Revenue < 1200.**

For **Shop 0** (the one with little data), calculate the probability that it is "Critical" based on your posterior samples.

$$ P(\theta_0 < 1200 | \text{Data}) $$

In [ ]:
# Your analysis here
# prob_critical = ...

### Question 3
Suppose Shop 0 had a `raw_mean` of 1150 (which is < 1200). 
If you used the Naive approach (No Pooling), you might close it immediately.
Does your Bayesian model agree? What does the "borrowing of strength" imply about Shop 0's potential?

*(Write your answer here)*